In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, glob, os
import numpy as np
import pandas as pd
import helpers
from prompts import *

# Important Paremeters

In [ ]:
path = "~/kg_aug_causal_disc_exp"

In [ ]:
# variables of interest
with open("variable_definitions/default_definitions.json", "r") as file:
    def_map = json.load(file)

# Markdown parsing and chunking

In [ ]:
import markdown_parser
from pathlib import Path
from config import DIRECTORY

chunks = []
files = Path(DIRECTORY).glob('**/*.md')
for file in files:
    print(file)
    if os.path.isfile(file):
        # simple markdown parser that removes citations, urls, references, acknowledgements, and basically everything after the conclusion
        content = markdown_parser.process_markdown_paper(str(file))
        # semantic chunk is chunking w.r.t sentences, and has overlap param as well
        chunks.extend(markdown_parser.semantic_chunk(content) )

# Creating the Retriever

In [ ]:
model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')

In [ ]:
embs  = model.encode(chunks, convert_to_numpy=True)

In [ ]:
norms = np.linalg.norm(embs, axis=1, keepdims=True)        # shape (N, 1)
embs_normalized = embs / np.clip(norms, a_min=1e-12, a_max=None)

In [ ]:
dim   = embs_normalized.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embs_normalized)

In [ ]:
import numpy as np
from config import query_context_window
import helpers

# we aren't actually using k here
def get_k_docs(query: str, k: int = 100):
    # 1. Embed the query
    q_emb = model.encode([query], convert_to_numpy=True)  # shape (1, D)
    
    # 2. Search the FAISS index
    #    D: array of squared L2 distances, shape (1, k)
    #    I: array of indices of nearest neighbors, shape (1, k)
    D, I = index.search(q_emb, k)
    
    # 3. Fetch the top-k documents
    results = []
    token_count = 0
    for dist, idx in zip(D[0], I[0]):
        if helpers.token_count(chunks[idx] + "\n") + token_count > query_context_window:
            break
        token_count += helpers.token_count(chunks[idx] + "\n")
        results.append(chunks[idx]) # the original text or metadata
        

    return "\n".join(results)

In [ ]:
def retrieve_context(var1, var2, debug=False):

    var1res = get_k_docs(f"{def_map.get(var1, var1)}")
    var2res = get_k_docs(f"{def_map.get(var2, var2)}")
    
    final_report = f"# Report for Variable 1: {var1}\n" + var1res + f"\n# Report for Variable 2: {var2}\n" + var2res
    if debug:
        print(final_report)
    
    return final_report

# Setting Up RAG iterators

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

class Reasoning_Step(BaseModel):
    reasoning_step: str = Field(..., description="An intermediate reasoning step for breaking down the given context and query")

class Answer(BaseModel):
    reasoning: List[Reasoning_Step] = Field(..., description="List of reasoning steps")
    conclusion: Literal['A', 'B', 'C']

In [ ]:
from llm_client import get_client
generator = get_client(schema=Answer)

In [ ]:
from promptsd.causal_literature_prompts import reduce_rag_causal_lit

def local_retriever(query, var1, var2, summary, debug=False):
    if debug:
        print(reduce_rag_causal_lit(query, var1, var2, summary, def_map))
    response = generator(reduce_rag_causal_lit(query, var1, var2, summary, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})

    return response.conclusion, helpers.reasoning_to_string_multiple_choice(response)
    

In [ ]:
from promptsd.query_prompts import causal_lit_prompt

def query_local_causality(row):
    var1, var2, label = row['var1'], row['var2'], row["label"]
    # bandaid for now
    var1 = "Sleep disturbance" if var1 == "Sleep" else var1
    var2 = "Sleep disturbance" if var2 == "Sleep" else var2
        
    report = retrieve_context(var1, var2)
    clquery = causal_lit_prompt(var1, var2)
    causal_lit, clreasoning = local_retriever(clquery, var1, var2, report)
    return [var1, var2, causal_lit, clreasoning, report, label]

In [ ]:
proto = pd.read_csv(f"{path}/data/proto_cleaned.csv").drop(columns=["Unnamed: 0"])
full = pd.read_csv(f"{path}/data/full_cleaned.csv").drop(columns=["Unnamed: 0"])

# Setting Up the Experiment

In [ ]:
res = full.apply(query_local_causality, axis=1)

In [ ]:
columns = "Var1", "Var2", "Causal Literature", "Causal Literature Reasoning", "Report", "Label"
local_res = pd.DataFrame(res.to_list(), columns=columns)
local_res.to_csv("results/llm+rag_full_causal_literature.csv")
local_res